# Silver Layer — Transformation (Python / Pandas version)
**CNG Distribution Analytics Platform**

Reads Bronze **CSV** files, applies cleaning and business logic,
and writes clean **CSV** files to the Silver output folder under `Files/`.


| Step | Action |
|------|--------|
| 1 | Read Bronze CSV |
| 2 | Strip whitespace from column names + string cells |
| 3 | Cast date columns (handles mixed formats) |
| 4 | Normalise casing on categoricals |
| 5 | Coerce numeric columns |
| 6 | Normalise boolean / flag columns |
| 7 | Drop rows missing required keys |
| 8 | Deduplicate |
| 9 | Apply table-specific business logic |
| 10 | Stamp Silver audit column |
| 11 | Write Silver CSV |



## 1. Config

In [9]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────────
# FABRIC: notebook attached to a Lakehouse — read/write under Files/
BRONZE_PATH = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/" 
SILVER_PATH = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/"


# LOCAL: uncomment these two lines instead
# BRONZE_PATH = "data/output/bronze/"
# SILVER_PATH = "data/output/silver/"

os.makedirs(SILVER_PATH, exist_ok=True)

# ── Table configs ────────────────────────────────────────────────────────────
# required_cols : rows missing any of these are dropped
# date_cols     : parsed with the multi-format parser
# upper_cols    : trimmed + title-cased (first letter of each word; NaN preserved)
# code_cols     : trimmed + UPPER-cased (ISO codes / abbreviations stay all-caps)
# numeric_cols  : coerced to numbers (bad values -> NaN)
# bool_cols     : normalised to True/False from 0/1/Yes/No/true/false
TABLE_CONFIGS = {

    "bronze_purchase_orders": {
        "target"       : "silver_purchase_orders",
        "required_cols": ["POID", "ProductID", "SupplierID"],
        "date_cols"    : ["OrderDate", "ExpectedDeliveryDate", "ActualDeliveryDate"],
        "upper_cols"   : ["Status", "ApprovalStatus", "PortOfLoading"],
        "code_cols"    : ["IncoTerms", "Currency"],
        "numeric_cols" : ["OrderedQty", "ReceivedQty", "UnitCost_USD",
                          "TotalCost_USD", "FreightCost_USD", "ExchangeRate"],
        "bool_cols"    : [],
        "special"      : "purchase_orders"
    },

    "bronze_sales_orders": {
        "target"       : "silver_sales_orders",
        "required_cols": ["OrderID", "CustomerID", "ProductID"],
        "date_cols"    : ["OrderDate", "RequestedDeliveryDate"],
        "upper_cols"   : ["Status", "Division", "SalesChannel"],
        "code_cols"    : [],
        "numeric_cols" : ["OrderedQty", "UnitPrice_USD",
                          "LineTotal_USD", "DiscountPct"],
        "bool_cols"    : ["IsRush"],
        "special"      : "sales_orders"
    },

    "bronze_products": {
        "target"       : "silver_products",
        "required_cols": ["ProductID"],
        "date_cols"    : ["IntroducedDate"],
        "upper_cols"   : ["Category", "SubCategory"],
        "code_cols"    : ["UOM"],
        "numeric_cols" : ["BasePrice_USD", "StandardCost_USD",
                          "HSCode", "WeightPerUnit_KG"],
        "bool_cols"    : ["IsHazardous", "ActiveFlag"],
        "special"      : "products"
    },

    "bronze_customers": {
        "target"       : "silver_customers",
        "required_cols": ["CustomerID"],
        "date_cols"    : ["CustomerSince"],
        "upper_cols"   : ["Segment", "Region", "Division",
                          "AnnualRevenueTier", "PaymentTerms"],
        "code_cols"    : ["PreferredCurrency"],
        "numeric_cols" : ["CreditLimit_USD"],
        "bool_cols"    : ["ActiveFlag", "TaxExempt"],
        "special"      : "customers"
    },

    "bronze_inventory": {
        "target"       : "silver_inventory",
        "required_cols": ["InventoryID", "ProductID", "WarehouseID"],
        "date_cols"    : ["LastReceivedDate", "LastIssuedDate",
                          "ExpiryDate", "LastUpdated"],
        "upper_cols"   : [],
        "code_cols"    : [],
        "numeric_cols" : ["StockQty", "AllocatedQty", "AvailableQty",
                          "ReorderLevel", "ReorderQty",
                          "UnitCost_USD", "TotalValue_USD"],
        "bool_cols"    : ["BelowReorder"],
        "special"      : "inventory"
    }
}

print(f"Config loaded  : {len(TABLE_CONFIGS)} tables")
print(f"Bronze path    : {BRONZE_PATH}")
print(f"Silver path    : {SILVER_PATH}")

Config loaded  : 5 tables
Bronze path    : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/
Silver path    : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/


## 2. Imports

In [10]:
import pandas as pd
import numpy as np
from datetime import datetime

transformed_at = datetime.now()
print(f"pandas  : {pd.__version__}")
print(f"Run started : {transformed_at.strftime('%Y-%m-%d %H:%M:%S')}")

pandas  : 2.3.3
Run started : 2026-06-17 23:10:45


## 3. Helper functions
Robust date parsing and boolean normalisation — the source files mix
formats and flag styles, so these handle them explicitly.

In [11]:
# Known date formats present in the raw files, tried in order.
#   2023-12-04            -> %Y-%m-%d
#   08-16-2023            -> %m-%d-%Y
#   01/19/24              -> %m/%d/%y
#   09/03/2024            -> %m/%d/%Y
#   18 January 2013       -> %d %B %Y
#   2024-10-21 00:00:00   -> %Y-%m-%d %H:%M:%S
DATE_FORMATS = [
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%d",
    "%m-%d-%Y",
    "%m/%d/%Y",
    "%m/%d/%y",
    "%d %B %Y",
    "%d %b %Y",
]

def parse_mixed_dates(series: pd.Series) -> pd.Series:
    """
    Parse a column that mixes several date formats.
    Tries each known format on the still-unparsed values; whatever
    remains after all formats becomes NaT (instead of crashing).
    """
    s = series.astype(str).str.strip()
    s = s.replace({"": np.nan, "nan": np.nan, "NaN": np.nan, "None": np.nan})

    result = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    remaining = s.notna()

    for fmt in DATE_FORMATS:
        if not remaining.any():
            break
        parsed = pd.to_datetime(s[remaining], format=fmt, errors="coerce")
        good = parsed.notna()
        result.loc[parsed.index[good]] = parsed[good]
        # mark these as done
        remaining.loc[parsed.index[good]] = False

    # final catch-all for anything left (lets pandas guess)
    if remaining.any():
        parsed = pd.to_datetime(s[remaining], errors="coerce")
        result.loc[parsed.index] = parsed

    return result


TRUE_TOKENS  = {"1", "true", "yes", "y", "t"}
FALSE_TOKENS = {"0", "false", "no", "n", "f"}

def to_bool(series: pd.Series) -> pd.Series:
    """
    Normalise mixed flag styles (0/1, Yes/No, true/false, with stray
    whitespace/case) into proper booleans. Unknown -> NaN.
    """
    s = series.astype(str).str.strip().str.lower()
    mapped = s.map(lambda v: True if v in TRUE_TOKENS
                   else False if v in FALSE_TOKENS
                   else pd.NA)
    # Use pandas nullable 'boolean' dtype so True/False/NA coexist safely.
    # A plain object column mixing python bool + np.nan raises
    # 'boolean value of NA is ambiguous' during drop_duplicates / row hashing.
    return mapped.astype("boolean")

## 4. Generic cleaning function

In [12]:
def clean_dataframe(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Applies all generic cleaning steps.
    Runs on every table before any special logic.
    """
    original_rows = len(df)

    # 1. Strip whitespace from column names
    df.columns = df.columns.str.strip()

    # 2. Strip whitespace from every text cell.
    #    NOTE: with dtype=str, pandas 3.x labels columns 'str' (not 'object'),
    #    so a dtype==object test would skip them and leave IDs padded —
    #    breaking every downstream join. Detect text columns robustly instead.
    text_cols = [c for c in df.columns
                 if pd.api.types.is_object_dtype(df[c])
                 or pd.api.types.is_string_dtype(df[c])]
    for col in text_cols:
        df[col] = df[col].astype("string").str.strip()
    # turn empties / literal 'nan' strings into real NaN
    df = df.replace({"": np.nan, "nan": np.nan, "NaN": np.nan, "None": np.nan})

    # 3. Parse date columns (mixed formats handled in helper)
    for col in config["date_cols"]:
        if col in df.columns:
            df[col] = parse_mixed_dates(df[col])

    # 4. Normalise casing on categorical columns
    #    e.g. 'approved', 'APPROVED', ' Approved ' -> 'Approved'
    #    title case capitalises the first letter of each word
    for col in config["upper_cols"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()
            df[col] = df[col].replace({"Nan": np.nan, "None": np.nan, "": np.nan})

    # 4b. Codes / abbreviations stay UPPER-cased (currencies, UOM, IncoTerms)
    for col in config.get("code_cols", []):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
            df[col] = df[col].replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})

    # 5. Coerce numeric columns
    for col in config["numeric_cols"]:
        if col in df.columns:
            # Coerce to numpy float64 (not pandas nullable) so that NaN
            # comparisons like (price > 0) yield False rather than pd.NA,
            # which np.where / boolean masks cannot evaluate.
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

    # 6. Normalise boolean / flag columns
    for col in config.get("bool_cols", []):
        if col in df.columns:
            df[col] = to_bool(df[col])

    # 7. Drop rows missing required key fields
    before_drop = len(df)
    present_keys = [c for c in config["required_cols"] if c in df.columns]
    df = df.dropna(subset=present_keys)
    dropped_nulls = before_drop - len(df)

    # 8. Deduplicate
    #    Dedup on a string view of the row so nullable dtypes (boolean/Int
    #    with pd.NA) can't raise 'boolean value of NA is ambiguous' during
    #    row hashing. We keep the first occurrence in the original frame.
    before_dedup = len(df)
    keep_mask = ~df.astype(str).duplicated()
    df = df[keep_mask]
    dropped_dupes = before_dedup - len(df)

    # 9. Drop Bronze audit columns — Silver will add its own
    bronze_cols = ["_ingested_at", "_source_file", "_source_system", "_pipeline_run_id"]
    df = df.drop(columns=[c for c in bronze_cols if c in df.columns])

    # 10. Add Silver audit column
    df["_silver_transformed_at"] = str(transformed_at)

    print(f"  Raw rows        : {original_rows:,}")
    print(f"  Dropped (nulls) : {dropped_nulls:,}")
    print(f"  Dropped (dupes) : {dropped_dupes:,}")
    print(f"  Final rows      : {len(df):,}")

    return df

## 5. Table-specific business logic

In [13]:
# Canonical status maps — keys match the title-cased output of upper_cols
PO_STATUS_MAP = {
    "Fully Received": "Fully Received", "Fullyreceived": "Fully Received",
    "Partially Received": "Partially Received",
    "Open": "Open", "On Hold": "On Hold", "Cancelled": "Cancelled",
}
SO_STATUS_MAP = {
    "Delivered": "Delivered", "Deliverd": "Delivered",
    "Shipped": "Shipped", "Partially Shipped": "Partially Shipped",
    "Processing": "Processing", "Procesing": "Processing",
    "Confirmed": "Confirmed", "On Hold": "On Hold", "Cancelled": "Cancelled",
}
# Sales divisions — collapse casing/whitespace variants to canonical labels
DIVISION_MAP = {
    "North American Distribution": "North American Distribution",
    "Central National": "Central National",
    "Lindenmeyr Publications": "Lindenmeyr Publications",
}
# Country variants -> region (covers all spellings seen in Customers.csv)
COUNTRY_TO_REGION = {
    "USA": "AMER", "US": "AMER", "U.S.": "AMER", "U.S.A": "AMER",
    "UNITED STATES": "AMER",
    "CANADA": "AMER", "CAN": "AMER", "CA": "AMER",
    "MEXICO": "AMER", "BRAZIL": "AMER", "BRA": "AMER",
    "UK": "EMEA", "U.K.": "EMEA", "GB": "EMEA",
    "GERMANY": "EMEA", "DEUTSCHLAND": "EMEA", "FRANCE": "EMEA",
    "CHINA": "APAC", "JAPAN": "APAC", "AUSTRALIA": "APAC",
}


def apply_special_logic(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """
    Table-specific transforms that can't be generalised.
    Only tables that need extra logic are handled here.
    """

    # ── Purchase Orders ────────────────────────────────────────────────
    if table_name == "purchase_orders":

        # Canonicalise status text
        if "Status" in df.columns:
            df["Status"] = df["Status"].map(PO_STATUS_MAP).fillna(df["Status"])

        # Supplier lead times in days
        df["actual_lead_time_days"] = (
            df["ActualDeliveryDate"] - df["OrderDate"]
        ).dt.days
        df["expected_lead_time_days"] = (
            df["ExpectedDeliveryDate"] - df["OrderDate"]
        ).dt.days

        # Late delivery flag — only where ActualDeliveryDate exists (open POs stay NaN)
        df["is_late_delivery"] = np.where(
            df["ActualDeliveryDate"].notna(),
            df["ActualDeliveryDate"] > df["ExpectedDeliveryDate"],
            np.nan
        )

        # Receipt rate — how much of what was ordered actually arrived
        df["receipt_rate"] = np.where(
            df["OrderedQty"] > 0,
            (df["ReceivedQty"] / df["OrderedQty"]).round(4),
            np.nan
        )

        print("  [PO] status normalised, lead time, receipt rate, late flag added")

    # ── Sales Orders ──────────────────────────────────────────────────
    elif table_name == "sales_orders":

        # Canonicalise status + division text
        if "Status" in df.columns:
            df["Status"] = df["Status"].map(SO_STATUS_MAP).fillna(df["Status"])
        if "Division" in df.columns:
            df["Division"] = df["Division"].map(DIVISION_MAP).fillna(df["Division"])

        # Negative quantities are returns/credits — flag, don't delete
        df["is_return"] = df["OrderedQty"] < 0

        # Discounted unit price (DiscountPct is a percentage, e.g. 13.8)
        df["DiscountPct"] = df["DiscountPct"].fillna(0)
        df["discounted_unit_price"] = (
            df["UnitPrice_USD"] * (1 - df["DiscountPct"] / 100)
        ).round(4)

        # Recompute a clean line total from qty * discounted price
        df["calc_line_total_usd"] = (
            df["OrderedQty"].abs() * df["discounted_unit_price"]
        ).round(2)

        # NOTE: 261 null Divisions remain. SourceSystem (CRM/EDI/Web Portal/
        # SQL_Server_Sales) is a channel, NOT a division, so it cannot fill
        # Division. They are left as NaN for Gold to resolve via Customer.
        n_null_div = df["Division"].isna().sum() if "Division" in df.columns else 0
        print(f"  [SO] status/division normalised, returns flag, discounted price added")
        print(f"  [SO] {n_null_div} rows still missing Division — resolve in Gold via Customer join")

    # ── Products ─────────────────────────────────────────────────────
    elif table_name == "products":

        # Margin = (price - cost) / price
        if all(c in df.columns for c in ["BasePrice_USD", "StandardCost_USD"]):
            df["gross_margin_pct"] = np.where(
                df["BasePrice_USD"] > 0,
                ((df["BasePrice_USD"] - df["StandardCost_USD"])
                 / df["BasePrice_USD"] * 100).round(2),
                np.nan
            )
            # Flag products priced below cost (negative margin).
            # NaN-safe: unknown margin -> False, not an ambiguous NA.
            df["is_below_cost"] = (df["gross_margin_pct"] < 0).fillna(False)
            print("  [PR] gross margin %, below-cost flag added")

    # ── Customers ───────────────────────────────────────────────────
    elif table_name == "customers":

        # Fix Region/Country mismatches (US customers tagged APAC etc.)
        if "Country" in df.columns:
            corrected = (df["Country"].astype(str).str.upper().str.strip()
                         .map(COUNTRY_TO_REGION))
            # prefer the country-derived region; fall back to existing
            df["Region"] = corrected.combine_first(df.get("Region"))
            unresolved = corrected.isna().sum()
            print(f"  [CU] Region corrected from Country lookup"
                  f" ({unresolved} countries unmapped)")

        # Normalise AnnualRevenueTier casing (platinum/Bronze -> PLATINUM/BRONZE
        # already handled by upper_cols)

    # ── Inventory ──────────────────────────────────────────────────
    elif table_name == "inventory":

        # AllocatedQty has nulls -> treat missing as 0 allocated
        if "AllocatedQty" in df.columns:
            df["AllocatedQty"] = df["AllocatedQty"].fillna(0)

        # Recompute AvailableQty = StockQty - AllocatedQty (validation)
        if all(c in df.columns for c in ["StockQty", "AllocatedQty", "AvailableQty"]):
            calc = (df["StockQty"] - df["AllocatedQty"]).clip(lower=0)
            mismatch = (df["AvailableQty"] != calc).sum()
            if mismatch > 0:
                print(f"  [INV] {mismatch} AvailableQty mismatches found — recomputed")
            df["AvailableQty"] = calc

        # Recompute BelowReorder from StockQty vs ReorderLevel
        if all(c in df.columns for c in ["StockQty", "ReorderLevel"]):
            df["BelowReorder"] = df["StockQty"] <= df["ReorderLevel"]
            print("  [INV] BelowReorder flag recomputed from StockQty vs ReorderLevel")

    return df

## 6. Write function
Writes **CSV** to the Silver `Files/` folder.
Swap this one function when migrating back to full Spark/Delta in Fabric.

In [14]:
def write_silver(df: pd.DataFrame, target: str):
    """
    PYTHON VERSION  — writes CSV to Silver output folder under Files/.

    SPARK VERSION   — replace body with:
        spark_df = spark.createDataFrame(df)
        spark_df.write.format('delta').mode('overwrite') \\
            .option('overwriteSchema', 'true').saveAsTable(target)
    """
    out_path = SILVER_PATH + target + ".csv"
    df.to_csv(out_path, index=False)
    print(f"  Written to: {out_path}")

## 7. Main run — process all tables

In [15]:
results = []

for source_table, config in TABLE_CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"Processing: {source_table}")
    print(f"{'='*55}")

    try:
        # Read from Bronze CSV — dtype=str keeps everything raw for cleaning
        bronze_path = BRONZE_PATH + source_table + ".csv"
        df = pd.read_csv(bronze_path, dtype=str)
        print(f"  Read {len(df):,} rows from Bronze")

        # Generic cleaning
        df = clean_dataframe(df, config)

        # Table-specific business logic
        if config["special"]:
            df = apply_special_logic(df, config["special"])

        # Write Silver CSV
        write_silver(df, config["target"])

        results.append({
            "source" : source_table,
            "target" : config["target"],
            "status" : "SUCCESS",
            "rows"   : len(df),
            "error"  : None
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({
            "source" : source_table,
            "target" : config["target"],
            "status" : "FAILED",
            "rows"   : 0,
            "error"  : str(e)
        })

print(f"\n{'='*55}")
print("SILVER TRANSFORMATION COMPLETE")
print(f"{'='*55}")


Processing: bronze_purchase_orders
  Written to: abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/silver_purchase_orders.csv

Processing: bronze_sales_orders
  Read 5,150 rows from Bronze
  Raw rows        : 5,150
  Dropped (nulls) : 148
  Dropped (dupes) : 138
  Final rows      : 4,864
  [SO] status/division normalised, returns flag, discounted price added
  [SO] 0 rows still missing Division — resolve in Gold via Customer join
  Written to: abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/silver_sales_orders.csv

Processing: bronze_products
  Read 42 rows from Bronze
  Raw rows        : 42
  Dropped (nulls) : 0
  Dropped (dupes) : 2
  Final rows      : 40
  [PR] gross margin %, below-cost flag added
  Written to: abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/silver_products.csv

Processing: bronze_customers
  Read 72 rows from Bronze
  Ra

/tmp/ipykernel_230/2071818148.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s[remaining], errors="coerce")


## 8. Summary

In [16]:
summary = pd.DataFrame(results)
print(summary[["source", "target", "status", "rows", "error"]].to_string(index=False))

failed = summary[summary["status"] == "FAILED"]
if len(failed) > 0:
    raise Exception(f"{len(failed)} table(s) failed. Do not proceed to Gold.")
else:
    print("\nAll tables transformed. Safe to run Gold notebook.")

                source                 target  status  rows error
bronze_purchase_orders silver_purchase_orders SUCCESS  1144  None
   bronze_sales_orders    silver_sales_orders SUCCESS  4864  None
       bronze_products        silver_products SUCCESS    40  None
      bronze_customers       silver_customers SUCCESS    70  None
      bronze_inventory       silver_inventory SUCCESS   187  None

All tables transformed. Safe to run Gold notebook.


## 9. Quick validation

In [17]:
# Spot-check each Silver CSV — row counts and key derived columns
checks = {
    "silver_purchase_orders": ["actual_lead_time_days", "receipt_rate", "is_late_delivery"],
    "silver_sales_orders"   : ["discounted_unit_price", "is_return", "calc_line_total_usd"],
    "silver_products"       : ["gross_margin_pct", "is_below_cost"],
    "silver_customers"      : ["Region", "Segment"],
    "silver_inventory"      : ["AvailableQty", "BelowReorder"]
}

for target, cols_to_check in checks.items():
    path = SILVER_PATH + target + ".csv"
    try:
        df_check = pd.read_csv(path)
        print(f"\n{target}")
        print(f"  Rows    : {len(df_check):,}")
        print(f"  Columns : {len(df_check.columns)}")
        available = [c for c in cols_to_check if c in df_check.columns]
        if available:
            print("  Sample derived cols:")
            print(df_check[available].head(3).to_string())
    except Exception as e:
        print(f"{target}: ERROR — {e}")


silver_purchase_orders
  Rows    : 1,144
  Columns : 24
  Sample derived cols:
   actual_lead_time_days  receipt_rate  is_late_delivery
0                   46.0           1.0               1.0
1                   77.0           1.0               0.0
2                    NaN           0.0               NaN

silver_sales_orders
  Rows    : 4,864
  Columns : 21
  Sample derived cols:
   discounted_unit_price  is_return  calc_line_total_usd
0              2379.9992      False           2967859.00
1              2485.3524       True           3439727.72
2              2383.8903      False            727086.54

silver_products
  Rows    : 40
  Columns : 17
  Sample derived cols:
   gross_margin_pct  is_below_cost
0            -18.04           True
1             23.55          False
2             27.80          False

silver_customers
  Rows    : 70
  Columns : 18
  Sample derived cols:
  Region              Segment
0   AMER  Packaging Converter
1   AMER   Commercial Printer
2   AMER        